# GenAI In Action Chapter 07 Text Generation

## Connect to OpenAI API

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# define the environment path. it is at the parent folder
env_path ="../.env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

## List all models available in OpenAI API

In [ ]:
models=client.models.list()

# Print out the names of all the available models
for model in models:
    print("ID:", model.id)
    print("-------------------")

## Completion API

In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt="Write a few bullets on why pets are so awesome ",
    max_tokens=100,
    temperature=0.8
)
print(response.choices[0].text.strip())

In [ ]:
prompt = """Suggest three names for a new pet salon business.
The generated name ideas should evoke positive emotions and the
following key features: Professional, friendly, Personalized Service."""

response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    max_tokens=100,
    temperature=0.8,
    stop=None
)
print(response.choices[0].text.strip())

In [ ]:
# full output, how many tokens were used?
response

### Multiple completion

In [ ]:
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    temperature=0.8,
    max_tokens=100,
    n=3,
    stop=None)

for choice in response.choices:
    print(choice.text)

In [ ]:
# check the full output. How many tokens were used?
response

### Best of multiple output

In [ ]:
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    temperature=0.7,
    max_tokens=100,
    best_of=5,
    stop=None)

for choice in response.choices:
    print(choice.text)

In [ ]:
response

### Controlling randomness using temperature

In [ ]:
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    temperature=0.2,   # change the temperature to 0.2
    max_tokens=100,
    stop=None)

for choice in response.choices:
    print(choice.text)

In [ ]:
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    temperature=1.8,   # change the temperature, not recommended to set this high
    max_tokens=100,
    n=3,
    stop=None)

for choice in response.choices:
    print(choice.text)

Controlling randomness using top_p

In [ ]:
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
   # temperature=0.2,   # change the temperature to 0.2
    max_tokens=100,
    top_p=0.8,
    stop=None)

for choice in response.choices:
    print(choice.text)

## Chat Completion API

In [ ]:
# Call OpenAI Chat Completion API
response = client.chat.completions.create(
    model="gpt-3.5-turbo",  
    messages=[
        {"role": "system", "content": "You are a helpful assistant that helps people find information."},
        {"role": "user", "content":"Hello World"},
        {"role": "assistant", "content":"Hello! How can I assist you today?"},
        {"role": "user", "content": "I want to know more about pets and why dogs are good for humans?"}],
    temperature=0.8,
    max_tokens=800, 
    user="suhong",
    top_p=0.95
)

print(response.choices[0].message.content)

In [ ]:
response

### Rhyming Assistant

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",  
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant that provides truthful information. You answer all questions in rhyme."},
        {"role": "user", "content":"Hello World"},
        {"role": "assistant", "content":"Hello! How can I assist you today?"},
        {"role": "user", "content": "Who are the founders of Microsoft?"}],
    temperature=0.8,
    max_tokens=800, 
    user="suhong",
    top_p=0.95
)

print(response.choices[0].message.content)

## Chat completion as a completion API example

In [ ]:
# Call OpenAI Chat Completion API
response = client.chat.completions.create(
    model="gpt-3.5-turbo",  
    messages=[
        {"role": "system", "content": "You are a helpful assistant that helps people find information."},
        {"role": "user", "content":"Translate the following English text into Chinese: 'it is better to teach a man how to \
                                    fish than give him a fish' "}]
)

print(response.choices[0].message.content)

In [ ]:
# Call OpenAI Chat Completion API as a completion API Example

response = client.chat.completions.create(
    model="gpt-3.5-turbo",  
    messages=[
        {"role": "system", "content": "You are a helpful assistant that helps people learn to code."},
        {"role": "user", "content": "I need to write a Python function"},
        {"role": "user", "content": "This function should take two numbers as input and return their sum"}]
)

print(response.choices[0].message.content)

## ChatApp About Pets

We will develop a demo chatbot that can answer any pet-related questions and display the number of tokens used in each conversation.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken
import os

# define the environment path. it is at the parent folder
env_path ="../.env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

GPT_MODEL = "gpt-3.5-turbo"

system_content=system_content = """You are a helpful assistant that only answers questions related to pets.
                                For any other topic, respond with: "This app is only designed to answer questions about pets."""
system_message = {"role": "system", "content": system_content }
max_response_tokens = 250
token_limit = 4096
conversation = [system_message]

def num_tokens_from_messages(messages):
    encoding = tiktoken.get_encoding("cl100k_base")
    num_tokens = 0
    for message in messages:
        num_tokens += 4
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens -= 1
    num_tokens += 2
    return num_tokens

print("I am a helpful assistant. I can talk about any questions related to pets.")
print("Type 'exit' or 'quit' to end the conversation.\n")

while True:
    user_input = input("> ")
    if user_input.strip().lower() in {"exit", "quit"}:
        print("Goodbye!")
        break

    conversation.append({"role": "user", "content": user_input})
    conv_history_tokens = num_tokens_from_messages(conversation)

    while conv_history_tokens + max_response_tokens >= token_limit:
        del conversation[1]  # Don't delete the system message
        conv_history_tokens = num_tokens_from_messages(conversation)

    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=conversation,
        temperature=0.8,
        max_tokens=max_response_tokens
    )

    assistant_message = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": assistant_message})

    print("\n" + assistant_message)
    print(f"(Tokens used: {response.usage.total_tokens})\n")